# PBL4 — Tự train biểu cảm bằng FER+ trên Colab

Nhánh: `thai+emotion-recognition-training`.

**A**: model FER+ ONNX đang dùng. **B**: CNN mới, trọng số ngẫu nhiên, nhãn cứng.
**C**: cùng CNN, nhãn mềm. Giữ nguyên model hiện tại để so sánh.

Notebook này chưa chứa kết quả train thực. Chạy kiểm thử code bằng ảnh nhân tạo
không chứng minh accuracy trên FER+ hoặc webcam.

Trong Colab: **Runtime → Change runtime type → GPU**. Chạy từng mục theo thứ tự.
Dataset, checkpoint, báo cáo lưu ở Drive; dữ liệu train được chép vào máy Colab
để tránh đọc hàng nghìn file qua Drive. Phiên bị ngắt: kết nối lại, chạy các ô
thiết lập và train; chương trình tiếp tục từ cuối epoch đã lưu.


## 1. Lấy code đúng nhánh và cài môi trường train riêng

Repository: https://github.com/Thainguyen2103/Panda-Robotics-Client-

Nếu chạy Jupyter trên máy khác, đặt `REPO` tới bản clone của bạn. Nếu repository
cần đăng nhập, tự clone theo quyền GitHub của bạn; không ghi token vào notebook.
Code chỉ được clone khi thư mục chưa tồn tại; ô này không tự reset code đã sửa.


In [ ]:
from pathlib import Path
import importlib.metadata
import json
import os
import subprocess
import sys

BRANCH = 'thai+emotion-recognition-training'
REPO = Path('/content/PBL4') if Path('/content').exists() else Path.cwd()
if not (REPO / '.git').is_dir():
    subprocess.run(['git', 'clone', '--single-branch', '--branch', BRANCH,
                    'https://github.com/Thainguyen2103/Panda-Robotics-Client-.git', str(REPO)], check=True)
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
actual_branch = subprocess.check_output(['git', 'branch', '--show-current'], text=True).strip()
if actual_branch != BRANCH:
    raise RuntimeError(f'Đang ở nhánh {actual_branch}, cần {BRANCH}.')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r',
                'training/emotion/requirements.txt'], check=True)
print('Code:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())

def run_module(module, *args):
    subprocess.run([sys.executable, '-m', module, *map(str, args)], cwd=REPO, check=True)


## 2. Nơi lưu dữ liệu và cấu hình thí nghiệm

Mặc định chạy một seed để kiểm tra luồng và chỉnh bằng validation. Khi đã chốt
cấu hình, đổi `SEEDS` thành `[42, 123, 2026]`, giữ cùng cấu hình cho B/C.
Đổi `EXPERIMENT` khi thay hyperparameter; không ghi đè một lượt thử cũ.


In [ ]:
try:
    from google.colab import drive
except ImportError:
    DRIVE_ROOT = Path.home() / 'PBL4-FERPlus'
else:
    drive.mount('/content/drive')
    DRIVE_ROOT = Path('/content/drive/MyDrive/PBL4-FERPlus')

RAW = DRIVE_ROOT / 'raw'
RUNS = DRIVE_ROOT / 'runs'
EXPORTS = DRIVE_ROOT / 'exports'
BASELINE_MODEL = DRIVE_ROOT / 'models' / 'emotion-ferplus-8.onnx'
for folder in (RAW, RUNS, EXPORTS, BASELINE_MODEL.parent):
    folder.mkdir(parents=True, exist_ok=True)

EXPERIMENT = 'cnn-v1'
SEEDS = [42]  # Sau khi chốt cấu hình: [42, 123, 2026]
EPOCHS = 80
BATCH_SIZE = 128
PATIENCE = 12
LR = 0.001
WEIGHT_DECAY = 0.0001
ALLOW_CPU = False  # Chỉ bật để thử luồng, không khuyến nghị train đầy đủ trên CPU.
import torch
if not torch.cuda.is_available() and not ALLOW_CPU:
    raise RuntimeError('Chưa có GPU. Hãy chọn GPU trong Runtime, kết nối lại và chạy từ đầu.')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Thiết bị:', torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'CPU')
print('Thư mục dữ liệu:', RAW)


## 3. Tải dataset và giữ đúng model đối chứng

**Ảnh gốc:** vào [trang FER2013 chính thức trên Kaggle](https://www.kaggle.com/c/challenges-in-representation-learning-facial-expression-recognition-challenge/data),
đăng nhập, đọc/chấp nhận điều kiện truy cập và tải dữ liệu. Giải nén để có
`fer2013.csv`, rồi tải file này lên **MyDrive/PBL4-FERPlus/raw/**.
Nếu tên file/tệp nén khác, tìm file CSV có hai cột `pixels`, `Usage`;
không thay bằng bộ thư mục FER bảy lớp đã mất liên kết dòng với FER+.

**Nhãn FER+:** ô kế tiếp tải `fer2013new.csv` từ
[Microsoft FERPlus](https://github.com/microsoft/FERPlus), không cần Kaggle token.
Nhãn được ghép theo thứ tự dòng, không theo nhãn bảy lớp cũ.

**Model đối chứng:** chép file hiện có tại
`C:\PBL4\models\vision\emotion-ferplus-8.onnx` lên
**MyDrive/PBL4-FERPlus/models/**. Git không chứa file trọng số này.
Notebook kiểm tra SHA256 để so sánh đúng file PBL4 đang sử dụng.

Không đưa CSV, ảnh mặt, checkpoint hoặc token vào repository.


In [ ]:
import urllib.request
from training.emotion.common import sha256

FER2013 = RAW / 'fer2013.csv'
FERPLUS = RAW / 'fer2013new.csv'
if not FERPLUS.exists():
    temporary = FERPLUS.with_suffix('.download')
    urllib.request.urlretrieve(
        'https://raw.githubusercontent.com/microsoft/FERPlus/master/fer2013new.csv', temporary)
    temporary.replace(FERPLUS)
if not FER2013.is_file():
    raise FileNotFoundError(f'Cần tải FER2013 từ Kaggle và đặt file tại: {FER2013}')
print('FER2013 SHA256:', sha256(FER2013))
print('FER+ SHA256:', sha256(FERPLUS))


## 4. Ghép ảnh–nhãn và kiểm tra dữ liệu

Giữ `Training → train`, `PublicTest → validation`, `PrivateTest → test`.
Giao thức **shared-v1** bỏ phiếu đơn lẻ, mẫu unknown/NF chiếm ưu thế, mẫu không
đủ phiếu biểu cảm và mẫu hòa nhãn đứng đầu. Chuẩn hóa tám phiếu biểu cảm thành
nhãn mềm; B dùng argmax trên đúng các ảnh C sử dụng. Đây là giao thức riêng,
không phải tái lập bộ lọc/kiến trúc CNTK của bài báo gốc.

Đọc số mẫu bị lọc và số mẫu mỗi lớp. Trùng ảnh giữa split được ghi trong
manifest; notebook không tự xáo trộn split để che mất vấn đề này.


In [ ]:
import shutil
from training.emotion.data import prepare, load_split

PREPARED_DRIVE = DRIVE_ROOT / 'prepared-shared-v1'
if not PREPARED_DRIVE.exists():
    local_new = Path('/content/ferplus-build') if Path('/content').exists() else DRIVE_ROOT / 'prepared-build'
    if local_new.exists():
        raise RuntimeError(f'Thư mục chuẩn bị cũ còn tồn tại: {local_new}. Chọn một thư mục mới.')
    prepare(FER2013, FERPLUS, local_new)
    shutil.copytree(local_new, PREPARED_DRIVE)
manifest = json.loads((PREPARED_DRIVE / 'manifest.json').read_text())
if manifest['sources'] != {'fer2013': sha256(FER2013), 'ferplus': sha256(FERPLUS)}:
    raise RuntimeError('Dữ liệu nguồn đã đổi. Dùng tên thư mục prepared và EXPERIMENT mới.')
DATA = Path('/content') / ('ferplus-' + manifest['fingerprint'][:12]) if Path('/content').exists() else PREPARED_DRIVE
if DATA != PREPARED_DRIVE:
    shutil.copytree(PREPARED_DRIVE, DATA, dirs_exist_ok=True)
for split in ('train', 'validation', 'test'):
    load_split(DATA, split)  # Verify saved file checksums.
print(json.dumps(manifest['splits'], indent=2))
print('Trùng ảnh giữa split:', len(manifest['cross_split_duplicate_groups']))
if manifest['has_cross_split_duplicates']:
    print('Cần ghi rõ hạn chế trùng ảnh trong báo cáo; không tuyên bố bộ chia không rò rỉ.')


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from training.emotion.common import LABELS

fig, ax = plt.subplots(figsize=(11, 4))
x = np.arange(8)
for offset, split in enumerate(('train', 'validation', 'test')):
    counts = [manifest['splits'][split]['class_counts'][label] for label in LABELS]
    ax.bar(x + (offset - 1) * .25, counts, .25, label=split)
ax.set_xticks(x, LABELS, rotation=25)
ax.set_ylabel('Số ảnh sau lọc')
ax.legend()
plt.tight_layout()
plt.show()

preview, preview_targets, _, _ = load_split(DATA, 'train')
fig, axes = plt.subplots(2, 4, figsize=(10, 5))
for index, ax in enumerate(axes.flat):
    if index < len(preview):
        ax.imshow(preview[index], cmap='gray', vmin=0, vmax=255)
        ax.set_title(LABELS[int(preview_targets[index].argmax())])
    ax.axis('off')
plt.tight_layout()
plt.show()


## 5. Đo model A trên validation trước khi train

Đây là phân loại trực tiếp ảnh mặt, không chạy YuNet/landmarks/luật thời gian.
Mọi model đều dùng cùng ảnh xám resize 64×64, đầu vào float32 0–255.
CNN mới đóng gói chuẩn hóa bên trong ONNX; không chuẩn hóa hai lần.

Ô này cần file model đối chứng ở Drive. Chỉ đo validation trong giai đoạn chọn
cấu hình. Tốc độ đo tại Colab là CPU Colab, không phải tốc độ máy PBL4.


In [ ]:
EXPECTED_BASELINE_SHA256 = 'a2a2ba6a335a3b29c21acb6272f962bd3d47f84952aaffa03b60986e04efa61c'
if not BASELINE_MODEL.is_file():
    raise FileNotFoundError(f'Chép model ONNX hiện tại vào {BASELINE_MODEL}')
if sha256(BASELINE_MODEL) != EXPECTED_BASELINE_SHA256:
    raise RuntimeError('File baseline khác model PBL4 đã ghi nhận. Kiểm tra nguồn trước khi tiếp tục.')
from training.emotion.evaluate import compare
import hashlib

def compare_saved(models, stage, split):
    signature = json.dumps({'dataset': manifest['fingerprint'], 'split': split,
                            'models': [(name, sha256(path)) for name, path in models]}, sort_keys=True)
    tag = hashlib.sha256(signature.encode()).hexdigest()[:12]
    folder = DRIVE_ROOT / 'reports' / EXPERIMENT / f'{stage}-{tag}'
    if (folder / 'comparison.json').is_file() and (folder / 'summary.csv').is_file():
        result = json.loads((folder / 'comparison.json').read_text())
        print('Dùng lại báo cáo đã đo:', folder, result['environment'])
    else:
        if folder.exists():
            raise RuntimeError(f'Báo cáo chưa ghi xong tại {folder}. Đổi EXPERIMENT trước khi đo lại.')
        result = compare(DATA, models, folder, split=split)
    return result, folder

_, BASELINE_REPORT = compare_saved([('pretrained', BASELINE_MODEL)], 'baseline-validation', 'validation')


## 6. Train B và C — chỉ chọn theo validation

Mỗi lượt có `best.pt`, `last.pt`, `history.json`, `config.json`, `status.json`.
Tiếp tục phiên dùng `last.pt`; xuất model dùng `best.pt`.
Kiểm tra config khi resume để tránh trộn dataset, seed hoặc learning rate.
Nếu đổi hyperparameter, đổi `EXPERIMENT` và chạy một thí nghiệm mới.


In [ ]:
RUN_DIRS = {}
for seed in SEEDS:
    for mode in ('hard', 'soft'):
        name = f'{mode}-seed{seed}'
        destination = RUNS / EXPERIMENT / name
        args = ['--data', DATA, '--output', destination, '--mode', mode,
                '--seed', seed, '--epochs', EPOCHS, '--batch-size', BATCH_SIZE,
                '--lr', LR, '--weight-decay', WEIGHT_DECAY, '--patience', PATIENCE,
                '--device', DEVICE]
        if (destination / 'last.pt').is_file():
            args.append('--resume')
        run_module('training.emotion.train', *args)
        RUN_DIRS[name] = destination
print('Hoàn thành:', list(RUN_DIRS))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for name, destination in RUN_DIRS.items():
    history = json.loads((destination / 'history.json').read_text())
    epochs = [row['epoch'] for row in history]
    axes[0].plot(epochs, [row['train']['loss'] for row in history], label=name + ' train')
    axes[0].plot(epochs, [row['validation']['loss'] for row in history], '--', label=name + ' val')
    axes[1].plot(epochs, [row['validation']['macro_f1'] for row in history], label=name)
axes[0].set_title('Loss — validation dùng nhãn mềm chung cho B/C')
axes[1].set_title('Validation macro-F1 (8 lớp)')
for ax in axes:
    ax.set_xlabel('Epoch')
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()


## 7. Xuất ONNX và kiểm tra với OpenCV

Xuất batch=1, opset 13, tám logits. Bắt buộc so số với PyTorch bằng cả
ONNX Runtime và OpenCV, trên ảnh validation cùng các mẫu pixel biên 0/255.
Chỉ tạo model chính thức khi parity qua. File `.onnx.json` ghi thứ tự nhãn,
chuẩn hóa, hash, seed, epoch và kết quả parity; luôn giữ cùng file ONNX.


In [ ]:
from training.emotion.export import export
MODELS = [('pretrained', BASELINE_MODEL)]
for name, destination in RUN_DIRS.items():
    model_path = EXPORTS / EXPERIMENT / f'{name}.onnx'
    if not model_path.exists():
        export(destination / 'best.pt', DATA, model_path)
    else:
        saved = json.loads(model_path.with_suffix('.onnx.json').read_text())
        if saved['checkpoint_sha256'] != sha256(destination / 'best.pt'):
            raise RuntimeError('Checkpoint đã đổi: dùng tên export/EXPERIMENT mới.')
    MODELS.append((name, model_path))


## 8. So A/B/C trên validation

Cùng backend OpenCV CPU, số thread, warmup và thứ tự ảnh. Macro-F1 tính trên
đủ tám lớp; nếu thiếu lớp, báo cáo đánh dấu không đầy đủ. Báo cáo chứa cả
confusion matrix, dự đoán từng ảnh, hash dữ liệu/model và độ trễ median/p95.
Chọn cấu hình tại đây; không sửa theo tập test.


In [ ]:
comparison, VALIDATION_REPORT = compare_saved(MODELS, 'all-validation', 'validation')
print((VALIDATION_REPORT / 'summary.csv').read_text())
for mode in ('hard', 'soft'):
    values = [m['metrics']['macro_f1'] for m in comparison['models'] if m['name'].startswith(mode + '-seed')]
    print(mode, 'macro-F1 mean =', np.mean(values),
          'sample std =', np.std(values, ddof=1) if len(values) > 1 else 'cần ít nhất 2 seed')

fig, axes = plt.subplots(1, len(comparison['models']), figsize=(6 * len(comparison['models']), 5), squeeze=False)
for ax, report in zip(axes.flat, comparison['models']):
    matrix = np.array(report['metrics']['confusion_matrix'])
    image = ax.imshow(matrix, cmap='Blues')
    ax.set_xticks(range(8), LABELS, rotation=90)
    ax.set_yticks(range(8), LABELS)
    ax.set_xlabel('Dự đoán')
    ax.set_ylabel('Nhãn kiểm định')
    ax.set_title(report['name'])
    fig.colorbar(image, ax=ax, fraction=.046)
plt.tight_layout()
plt.show()


## 9. Kiểm tra cuối cùng — chỉ bật sau khi chốt cấu hình

Mặc định không chạy test khi chọn Run all. Chỉ đặt `RUN_FINAL_TEST=True` sau
khi chốt kiến trúc, cách lọc, hyperparameter, seed và checkpoint theo validation.
Không dùng kết quả test để chỉnh rồi báo lại trên cùng tập là kiểm định độc lập.


In [ ]:
RUN_FINAL_TEST = False
if RUN_FINAL_TEST:
    _, TEST_REPORT = compare_saved(MODELS, 'final-test', 'test')
else:
    print('Chưa mở tập test để lựa chọn mô hình. Chỉ dùng validation.')


## 10. Mang về PBL4 và đánh giá camera riêng

Tải `.onnx` và `.onnx.json` tương ứng về một thư mục mới ở máy PBL4.
Mặc định ứng dụng vẫn dùng `emotion-ferplus-8.onnx`. Để thử file tự train,
mở PowerShell tại `C:\PBL4`, đặt đường dẫn thực rồi khởi động lại:

```powershell
$env:MOON_EMOTION_MODEL = 'C:\EmotionResults\soft-seed42.onnx'
.\start.bat
```

Kết thúc thử nghiệm: `Remove-Item Env:MOON_EMOTION_MODEL` rồi khởi động lại để
quay về model gốc. Biến môi trường chỉ áp dụng cho phiên và tiến trình con.

Đo trực tiếp A/B/C trên CPU của máy PBL4 bằng `python -m training.emotion.evaluate`
với cùng dữ liệu prepared và các file ONNX; độ trễ Colab không thay thế số đo này.

Đánh giá hệ thống bằng ảnh camera có nhãn riêng với
`tools/evaluate_emotions.py --model C:\EmotionResults\soft-seed42.onnx ...`.
Giữ nguyên detector, crop, landmarks và luật xác nhận khi so hai model; chia
theo người/phiên quay. Ảnh tĩnh không đo được độ giật nhãn và trễ xác nhận:
phần đó cần video có nhãn riêng, chưa được notebook này tự động đo.

Tự train không bảo đảm vượt pretrained. Ghi cả lớp yếu, trùng ảnh giữa split,
chênh lệch camera/dataset và độ dao động giữa seed. Kết quả mô tả biểu cảm
nhìn thấy, không xác nhận trạng thái tâm lý bên trong.

Nguồn: [FER+](https://github.com/microsoft/FERPlus),
[FER+ ONNX](https://github.com/onnx/models/tree/main/validated/vision/body_analysis/emotion_ferplus),
[Colab FAQ](https://research.google.com/colaboratory/faq.html).
